# Claim Documents: Direct Claude PDF via Ray on Serverless (v5)

Parallelized PDF → structured JSON using the **Foundation Model API DocumentContent
block** — the other half of the story from
[`claim_doc_ai_parse.ipynb`](claim_doc_ai_parse.ipynb).

**Why this path exists.** As of 2026-04, `ai_query` still does not accept PDF bytes
(Andrew Shieh, 2026-04-07: *"we don't currently have support for PDF input types with
ai_query for any model and don't expect it to be available in the very short term
~2wks"*), but Claude on Databricks FMAPI already accepts PDFs via the `document`
content block over the OpenAI-compatible endpoint. Skipping the parse step lets the
model see tables, seals, and handwritten fields directly.

**Why Ray.** A single-threaded Python driver serializes every FMAPI call. Ray fans
them out concurrently — and since these calls are network-bound, driver-local
concurrency is the bottleneck that matters.

> **Environment**: Notebook Environment -> Base environment -> `5` (Serverless v5,
> Ray preinstalled).
>
> **Ray topology note.** `ray.util.spark.setup_ray_cluster` is **not compatible
> with Serverless** — it touches `spark.sparkContext` which Spark Connect blocks
> (`JVM_ATTRIBUTE_NOT_SUPPORTED`). On Serverless we run Ray *locally on the driver*
> (`ray.init()`) and let the scheduler multiplex across driver cores. For a classic
> multi-node cluster, use `setup_ray_cluster(...)` instead — see
> `ai_functions/les_miserables_ray.ipynb`.

## 1. Setup

In [ ]:
%pip install --quiet ray[default] openai mlflow
%restart_python

In [ ]:
import json, os, base64, time
import pandas as pd
from time import perf_counter
import mlflow

CATALOG, SCHEMA, VOLUME = "shm", "genai", "raw_pdfs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
MODEL = "databricks-claude-sonnet-4"

TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
BASE_URL = f"{WORKSPACE_URL}/serving-endpoints"

user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/document_intelligence_eval")

# Assumes PDFs are already staged -- run claim_doc_ai_parse.ipynb first,
# or copy this snippet from that notebook to download the golden set.
display(spark.sql(f"LIST '{VOLUME_PATH}'"))

## 2. Start Ray on the driver

Serverless gives us one driver with N cores — no `setup_ray_cluster`. `ray.init()`
spins up a local Ray scheduler; `num_cpus` is inferred from the driver. Tune
task-level `num_cpus=` on `@ray.remote` to the effective concurrency you want
against FMAPI (network-bound, so oversubscription is fine).

In [ ]:
import ray

if ray.is_initialized():
    ray.shutdown()

ray.init(ignore_reinit_error=True, include_dashboard=False)
print(ray.cluster_resources())

## 3. Shared prompt + schema

Same prompt and JSON schema as `claim_doc_ai_parse.ipynb` — the only variable we're
changing between the two notebooks is the extraction path, not the contract.

In [ ]:
EXTRACTION_PROMPT = """Extract structured fields from this insurance document. Return null for missing fields.

RULES:
1. Placeholders are not values. Template text like "Insert Insurer name", "UNDERWRITER NAME REQUIRED", "Carrier A", "Grantee Organization", "CBO NAME" must return null — not the literal placeholder.
2. Primary certificate only. If the document contains multiple certificates, endorsements, or instruction pages, use the first/primary certificate.
3. Dates in MM/DD/YYYY when possible.

Fields: document_type, insurer_name, insured_name, policy_number, effective_date, expiration_date, coverage_types (list), total_amount, key_tables.
"""

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {"name": "claim_extraction", "schema": {
        "type": "object",
        "properties": {
            "document_type":   {"type": ["string", "null"]},
            "insurer_name":    {"type": ["string", "null"]},
            "insured_name":    {"type": ["string", "null"]},
            "policy_number":   {"type": ["string", "null"]},
            "effective_date":  {"type": ["string", "null"]},
            "expiration_date": {"type": ["string", "null"]},
            "coverage_types":  {"type": "array", "items": {"type": "string"}},
            "total_amount":    {"type": ["string", "null"]},
            "key_tables":      {"type": ["string", "null"]},
        },
        "required": ["document_type", "insurer_name", "insured_name", "policy_number",
                     "effective_date", "expiration_date", "coverage_types", "total_amount", "key_tables"],
    }},
}

## 4. Load PDFs and fan out with Ray

Each remote task reads a single PDF, base64-encodes it, and calls FMAPI with a
`document` content block. The JSON schema is passed via `response_format` so the
model returns typed output — not free-form text we'd have to re-parse.

**Pattern**: put the PDF bytes on the object store via `ray.put()` only if you have
large shared payloads. For one-PDF-per-task, pass bytes in the call directly — Ray's
scheduler colocates the task with the data automatically.

In [ ]:
files_df = (spark.read.format("binaryFile").load(f"{VOLUME_PATH}/*.pdf")
    .selectExpr("regexp_extract(path, '([^/]+)$', 1) AS filename", "content")
    .toPandas())
print(f"Loaded {len(files_df)} PDFs")


# num_cpus=0.25 oversubscribes: FMAPI calls are network-bound, so running 4x
# more tasks than cores is free throughput. Real cap is the endpoint's rate limit.
@ray.remote(num_cpus=0.25)
def extract_pdf(filename: str, pdf_bytes: bytes,
                token: str, base_url: str, model: str,
                prompt: str, schema: dict) -> dict:
    from openai import OpenAI

    t0 = time.perf_counter()
    try:
        client = OpenAI(api_key=token, base_url=base_url)
        data = base64.standard_b64encode(pdf_bytes).decode("utf-8")
        resp = client.chat.completions.create(
            model=model, max_tokens=4096,
            messages=[{"role": "user", "content": [
                {"type": "document",
                 "source": {"type": "base64",
                            "media_type": "application/pdf",
                            "data": data}},
                {"type": "text", "text": prompt},
            ]}],
            response_format=schema,
        )
        return {"filename": filename,
                "extracted": resp.choices[0].message.content,
                "error": None,
                "latency_s": time.perf_counter() - t0}
    except Exception as e:
        return {"filename": filename, "extracted": None,
                "error": str(e),
                "latency_s": time.perf_counter() - t0}


t0 = perf_counter()
futures = [
    extract_pdf.remote(r.filename, r.content, TOKEN, BASE_URL, MODEL,
                       EXTRACTION_PROMPT, RESPONSE_SCHEMA)
    for r in files_df.itertuples(index=False)
]
results = ray.get(futures)
wall_s = perf_counter() - t0

results_df = pd.DataFrame(results)
print(f"wall: {wall_s:.1f}s   docs/s: {len(results_df) / wall_s:.2f}   "
      f"errors: {results_df['error'].notna().sum()}")
display(results_df[["filename", "latency_s", "error"]])

In [ ]:
(spark.createDataFrame(results_df[["filename", "extracted", "latency_s"]])
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.fmapi_ray_extractions"))
display(spark.table(f"{CATALOG}.{SCHEMA}.fmapi_ray_extractions"))

## 5. Golden set + tolerant scorer

Identical to the `ai_parse_document` notebook — shared contract means shared
evaluation.

In [ ]:
import re

GENERIC_ACORD_COVERAGES = ["General Liability", "Automobile Liability", "Umbrella Liability", "Workers Compensation"]
CMS1500_PAYORS = ["Medicare", "Medicaid", "TRICARE", "CHAMPVA"]

golden_set = [
    {"filename": "acord_certificate.pdf",      "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": "Sample company", "insured_name": "Sample Insured Inc", "policy_number": "Sample no",
     "effective_date": "01/01/2010", "expiration_date": "01/01/2011", "coverage_types": GENERIC_ACORD_COVERAGES},
    {"filename": "acord_nyc_dcla.pdf",         "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": "US Underwriters Insurance Co", "insured_name": None, "policy_number": "ABCD1234567",
     "effective_date": "07/01/2025", "expiration_date": "06/30/2026",
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "acord_nyc_mome.pdf",         "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": "123456789",
     "effective_date": None, "expiration_date": None,
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "nyc_workerscomp_sample.pdf", "document_type": "NYS Workers Compensation Certificate",
     "insurer_name": "ABC Insurance Company", "insured_name": None, "policy_number": "E 1234567890",
     "effective_date": "07/01/2016", "expiration_date": "06/30/2017",
     "coverage_types": ["Workers Compensation"]},
    {"filename": "acord_nyc_dycd.pdf",         "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None,
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "acord_moval.pdf",            "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": "XX0111222333",
     "effective_date": None, "expiration_date": None,
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "acord_ny_ogs.pdf",           "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": "04/01/2011", "expiration_date": "04/01/2012",
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Umbrella Liability", "Workers Compensation"]},
    {"filename": "nyc_acord_sample.pdf",       "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None, "coverage_types": GENERIC_ACORD_COVERAGES},
    {"filename": "cms1500_claim_form.pdf",     "document_type": "CMS-1500 Health Insurance Claim Form",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None, "coverage_types": CMS1500_PAYORS},
    {"filename": "acord_tx.pdf",               "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None, "coverage_types": GENERIC_ACORD_COVERAGES},
]

EVAL_FIELDS = ["document_type", "insurer_name", "insured_name", "policy_number",
               "effective_date", "expiration_date", "coverage_types"]

def _norm(s):
    return str(s).lower().strip().replace("commercial ", "").replace("'", "").replace("  ", " ")

def _list_match(exp, ext):
    if not exp and not ext: return 1.0
    if not exp or not ext:  return 0.0
    e, x = [_norm(v) for v in exp], [_norm(v) for v in ext]
    matches = sum(1 for v in e if any(v in w or w in v for w in x))
    return max(0.0, matches / len(e) - min(0.2, 0.05 * max(0, len(x) - matches)))

def _parse_date(s):
    m = re.match(r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2,4})", str(s).strip())
    if not m: return None
    mo, d, y = int(m[1]), int(m[2]), int(m[3])
    y += 2000 if y < 50 else 1900 if y < 100 else 0
    return (mo, d, y)

def _date_match(exp, ext):
    e, x = _parse_date(exp), _parse_date(ext)
    return 1.0 if e and x and e == x else 0.0

def score_field(ext_v, exp_v, field=None):
    if exp_v is None and ext_v in (None, "", "<UNKNOWN>", "null"): return 1.0
    if exp_v is None or ext_v is None: return 0.0
    if isinstance(exp_v, list): return _list_match(exp_v, ext_v if isinstance(ext_v, list) else [])
    if field in ("effective_date", "expiration_date"): return _date_match(exp_v, ext_v)
    e, x = _norm(exp_v), _norm(ext_v)
    return 1.0 if e == x else 0.8 if (e in x or x in e) else 0.0

def score_extraction(extracted, expected):
    s = {f: score_field(extracted.get(f), expected.get(f), f) for f in EVAL_FIELDS}
    s["accuracy"] = sum(s.values()) / len(EVAL_FIELDS)
    return s

### Benchmark summary

In [ ]:
golden = {g["filename"]: g for g in golden_set}
rows = []
for _, r in results_df.iterrows():
    if not r["extracted"] or r["filename"] not in golden:
        continue
    try:
        sc = score_extraction(json.loads(r["extracted"]), golden[r["filename"]])
    except json.JSONDecodeError:
        continue
    rows.append({"filename": r["filename"], "latency_s": r["latency_s"], **sc})

bench_df = pd.DataFrame(rows)
spark.createDataFrame(bench_df).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.fmapi_ray_benchmark")

summary = pd.DataFrame({
    "mean_accuracy": [bench_df["accuracy"].mean()],
    "p50_latency_s": [bench_df["latency_s"].median()],
    "p95_latency_s": [bench_df["latency_s"].quantile(0.95)],
    **{f: [bench_df[f].mean()] for f in EVAL_FIELDS},
}).round(3)

with mlflow.start_run(run_name="fmapi_ray"):
    mlflow.log_param("model", MODEL)
    mlflow.log_param("approach", "fmapi_document_content_ray_local")
    mlflow.log_param("ray_topology", "driver_local")
    mlflow.log_param("num_cpus_per_task", 0.25)
    mlflow.log_metric("mean_accuracy", float(bench_df["accuracy"].mean()))
    mlflow.log_metric("wall_s", float(wall_s))
    mlflow.log_metric("docs_per_s", float(len(results_df) / wall_s))
    mlflow.log_metric("p50_latency_s", float(bench_df["latency_s"].median()))
    mlflow.log_metric("p95_latency_s", float(bench_df["latency_s"].quantile(0.95)))
    mlflow.log_table(bench_df, "benchmark_detail.json")

display(summary)

## 6. Shut down Ray

Release the local Ray scheduler.

In [ ]:
ray.shutdown()